# 60 — Sync gold into the clinician app database

**Direction:** Lakehouse gold (Delta) → SQL database in Fabric (the Rayfin app's store)

The Rayfin app's data service is backed by a **SQL database in Fabric**, provisioned by
`rayfin up` from the decorated entity classes in `rayfin/data/`. It is not the Lakehouse.
So gold has to be pushed across — and this notebook is that push.

### Why not just point the app at the Lakehouse?

You could: the Lakehouse SQL analytics endpoint is queryable. Two reasons not to.

1. **The app writes.** `ReviewDecision` records clinical decisions, and the Lakehouse SQL
   endpoint is read-only. An app that reads from one store and writes to another is a
   worse audit story than one that does both in a database built for it.
2. **Row-level security lives in Data API Builder**, generated from the entity decorators.
   That machinery binds to the SQL database. Bypassing it would mean re-implementing
   RLS somewhere else — which is how permission bugs happen.

Both stores stay in the same governed estate, in the same workspace, under the same
sensitivity label. Nothing leaves.

### What this notebook writes

Eight tables, in two tiers:

| Table | Tier | Contents |
|---|---|---|
| `PatientRisks` | patient-scoped | one row per pre-op patient — **now the full clinical record**, not eight columns |
| `RiskExplanations` | patient-scoped | the EBM's six ranked term contributions per patient |
| `ReviewDecisions` | patient-scoped | written by the app; this notebook never touches it |
| `CohortStats` | model-scoped | binned cohort distributions, so a number on screen has a denominator |
| `ProviderStats` | model-scoped | one row per provider: headcount, mean risk, flagged count, band counts (§2d) |
| `ModelCurves` | model-scoped | the calibration curve and the EBM shape functions |
| `ThresholdOptions` | model-scoped | the operating-point sweep, in patients per 1,000 |
| `ModelCards` | model-scoped | what the model is, measured on what, and whether the data is synthetic |

The patient-scoped tier filters every row against a token claim. The model-scoped tier
does not, and section 2b says why that is a decision rather than an omission.

### The `assignedClinicianId` mapping

Rayfin policies compare token claims (`sub`, `email`, `role`) against fields on the row.
There is no `provider_code` claim, so provider→clinician assignment is **materialised
here**, as a column. Edit `CLINICIAN_ASSIGNMENTS` below to match your demo accounts.

The column holds the Rayfin **app user id** — the bare GUID the app header shows.
Not the Entra object id, and not the full `/eid1/...` sub path: the backend strips
sub to its last segment before Data API Builder evaluates the policy.

## 1. Configuration

In [ ]:
GOLD_RISK    = "gold.patient_risk"
GOLD_EXPLAIN = "gold.risk_explanation"
# The clinical detail the app now shows lives here, not on the score table.
GOLD_COHORT  = "gold.knee_preop_cohort"

# --- Fabric SQL database created by `rayfin up` ----------------------------
# Find the connection string in the Fabric portal: open the SQL database item ->
# Settings -> Connection strings. Or: `fab get <workspace>.SQLDatabase/<name>`.
SQL_SERVER   = "kkbjqmtex52eljypc73n467eha-k5wrgg773jdu3bpwulh42qcxgy.database.fabric.microsoft.com"
SQL_DATABASE = "knee-risk-clinician-app-af0eb33e-b426-447d-bff6-577b3ed6390e"

# --- Which clinician sees which provider's patients ------------------------
# Each provider maps to one clinician. The value may be:
#
#   "first.last@trust.example"            an email address  (preferred — readable,
#                                         and knowable before anyone has signed in)
#   "7f775fb9-fdbc-458c-b8f7-e95f1b900f54"  a Rayfin app user id (a bare GUID)
#   ("<guid>", "<email>")                 both, if you have them
#
# The entity policy on PatientRisk accepts *either* key:
#
#   claims.sub   == assignedClinicianId
#   claims.email == assignedClinicianEmail
#
# Email is the one to use. The app user id is minted the first time that person
# signs in, so it cannot be filled in beforehand; an address can. The id branch
# stays because it is the path verified to work on 2026-08-31, and because
# whether Data API Builder receives an `email` claim for a Fabric-brokered
# session is a property of the platform, not of this notebook. Writing both
# columns costs nothing and means the demo works whichever claim arrives.
#
# To read your own values: sign in to the app, open devtools, and run
#   const k = Object.keys(localStorage).find(k => k.includes('authSession'));
#   const s = JSON.parse(localStorage.getItem(k)); console.log(s.user);
#
# The "*" entry is the catch-all for any provider not listed. While you are
# getting the demo running, point "*" at yourself so every patient is visible.
CLINICIAN_ASSIGNMENTS = {
    # Both keys for the presenter, so the row matches whichever claim Data API
    # Builder is handed. The GUID is the Rayfin app user id; the address is the
    # Entra sign-in. Catch-all, so every provider's patients are visible while
    # the demo is being set up.
    "*": ("7f775fb9-fdbc-458c-b8f7-e95f1b900f54",
          "audrius.saras@kpmgadvisorynl.onmicrosoft.com"),

    # Add the second demo clinician here to split the cohort for the RLS demo,
    # e.g.  "RGT": "second.clinician@kpmgadvisorynl.onmicrosoft.com",
}

# Sentinels, not empty strings. An empty string can collide with an absent claim
# and turn a mis-set row into a visible one; these two match nothing.
NO_ID    = "unassigned"
NO_EMAIL = "unassigned@invalid"

import re
_GUID  = re.compile(r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$")
_EMAIL = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")


def _split(value):
    """(app_user_id, email) for one configured value. Raises on anything unusable."""
    if isinstance(value, (tuple, list)):
        pair = [str(v).strip() for v in value]
        guid  = next((v for v in pair if _GUID.match(v)), NO_ID)
        email = next((v.lower() for v in pair if _EMAIL.match(v)), NO_EMAIL)
        if guid == NO_ID and email == NO_EMAIL:
            raise ValueError(f"Neither a GUID nor an email: {value!r}")
        return guid, email
    v = str(value).strip()
    if _GUID.match(v):
        return v, NO_EMAIL
    if _EMAIL.match(v):
        return NO_ID, v.lower()
    raise ValueError(
        f"{v!r} is neither an email address nor a Rayfin app user id (a bare 36-character "
        f"GUID). Placeholders left in CLINICIAN_ASSIGNMENTS are the usual cause."
    )


ASSIGNMENTS = {provider: _split(value) for provider, value in CLINICIAN_ASSIGNMENTS.items()}
if "*" not in ASSIGNMENTS:
    raise ValueError('CLINICIAN_ASSIGNMENTS needs a "*" catch-all, or unlisted providers are invisible to everyone.')

print("Assignments:")
for provider, (guid, email) in ASSIGNMENTS.items():
    print(f"  {provider:6s} id={guid:38s} email={email}")

import math
from pyspark.sql import functions as F
print("configured")


## 2. Shape gold to the app's entity schema

Column names must match the TypeScript entity fields exactly — Rayfin derives the SQL
column names from the decorated class properties, so `oksT0Score` in TypeScript is
`oksT0Score` in SQL. Renaming here rather than in the app keeps the entity definitions
readable.

### Why this now joins two gold tables

`gold.patient_risk` carries the *score*: eight identifying and triage columns plus the
probability. That was all the app could show, and it is not enough to review a patient
with — a clinician asked to agree or override a risk score for someone described by eight
facts is rubber-stamping, not exercising oversight.

The clinical detail all exists already, in `gold.knee_preop_cohort`: the OKS subscales,
the twelve individual comorbidities, the EQ-5D-3L profile, symptom duration, previous
surgery, living arrangements, disability and assisted completion. So the sync joins the
two on `episode_id` and carries them across.

Nothing is derived here that is not already a gold column, with one exception —
`providerType`, which collapses the mutually exclusive `university_hospital` and
`independent_hospital` flags into the single label the UI renders.

### Codings, and where they are guessed

Three columns are recoded to booleans, and the mapping is **1 = yes, 0 = no, anything
else = NULL**. `NULL` reaches the app as "not recorded" and renders as such — it is never
shown as "no". If your silver layer encodes these differently (some PROMs extracts use
1/2 rather than 1/0), the distribution printed below will show it immediately, and the
mapping is one edit. Check it rather than assuming it: a questionnaire item silently read
as "no previous surgery" is a wrong fact on a clinical screen.


In [ ]:
COMORBIDITY_COLS = [
    "heart_disease", "high_bp", "stroke", "circulation", "lung_disease", "diabetes",
    "kidney_disease", "nervous_system", "liver_disease", "cancer", "depression", "arthritis",
]

# Two expressions, one per key the policy accepts. Built the same way: start at
# the catch-all, then override per provider.
_star_id, _star_email = ASSIGNMENTS["*"]
assignment_id_expr = F.lit(_star_id)
assignment_email_expr = F.lit(_star_email)
for provider, (guid, email) in ASSIGNMENTS.items():
    if provider == "*":
        continue
    match = F.col("provider_code") == provider
    assignment_id_expr = F.when(match, F.lit(guid)).otherwise(assignment_id_expr)
    assignment_email_expr = F.when(match, F.lit(email)).otherwise(assignment_email_expr)


def yes_no(col):
    """1 -> True, 0 -> False, anything else -> NULL ('not recorded' in the app)."""
    return F.when(F.col(col) == 1, F.lit(True)).when(F.col(col) == 0, F.lit(False)).otherwise(F.lit(None).cast("boolean"))


def level_1_3(col):
    """EQ-5D-3L levels. Out-of-range codes (9 = missing in some extracts) become NULL."""
    return F.when(F.col(col).between(1, 3), F.col(col).cast("int")).otherwise(F.lit(None).cast("int"))


# The score table carries the probability; the cohort table carries the clinical detail.
# Inner join: a scored patient with no cohort row would be a patient the app cannot
# describe, and showing a risk score next to an empty record is worse than omitting them.
risk    = spark.table(GOLD_RISK)
cohort  = spark.table(GOLD_COHORT).drop("provider_code", "region", "age_band",
                                        "oks_t0_score", "comorbidity_count",
                                        "display_name", "surgery_scheduled_date")

joined = risk.join(cohort, on="episode_id", how="inner")

dropped = risk.count() - joined.count()
if dropped:
    print(f"WARNING: {dropped:,} scored patients have no row in {GOLD_COHORT} and were dropped. "
          "Re-run 30_gold_features and 50_batch_score against the same cohort.")

# Verify the boolean codings before trusting them -- see the markdown above.
print("Coding check (expect only 0, 1 and NULL):")
for c in ["t0_previous_surgery", "t0_disability", "t0_assisted"] + COMORBIDITY_COLS[:3]:
    if c in joined.columns:
        vals = [r[c] for r in joined.select(c).distinct().limit(10).collect()]
        print(f"  {c:24s} {sorted(v for v in vals if v is not None)}")

patient_risk = (joined
    .withColumn("assignedClinicianId", assignment_id_expr)
    .withColumn("assignedClinicianEmail", assignment_email_expr)
    .withColumn("providerType",
        F.when(F.col("independent_hospital") == 1, F.lit("independent_sector"))
         .when(F.col("university_hospital")  == 1, F.lit("university_hospital"))
         .otherwise(F.lit("nhs_trust")))
    .select(
        F.col("episode_id").alias("episodeId"),
        F.col("display_name").alias("displayName"),
        "assignedClinicianId",
        "assignedClinicianEmail",
        F.col("provider_code").alias("providerCode"),
        "providerType",
        "region",
        F.col("age_band").alias("ageBand"),
        "sex",
        # ---- Oxford Knee Score, total and the three subscales -----------------
        F.col("oks_t0_score").cast("int").alias("oksT0Score"),
        F.col("oks_pain_subscale").cast("int").alias("oksPainSubscale"),
        F.col("oks_function_subscale").cast("int").alias("oksFunctionSubscale"),
        F.col("oks_adl_subscale").cast("int").alias("oksAdlSubscale"),
        # ---- comorbidities: the count AND the twelve conditions ---------------
        F.col("comorbidity_count").cast("int").alias("comorbidityCount"),
        yes_no("heart_disease").alias("heartDisease"),
        yes_no("high_bp").alias("highBp"),
        yes_no("stroke").alias("stroke"),
        yes_no("circulation").alias("circulation"),
        yes_no("lung_disease").alias("lungDisease"),
        yes_no("diabetes").alias("diabetes"),
        yes_no("kidney_disease").alias("kidneyDisease"),
        yes_no("nervous_system").alias("nervousSystem"),
        yes_no("liver_disease").alias("liverDisease"),
        yes_no("cancer").alias("cancer"),
        yes_no("depression").alias("depression"),
        yes_no("arthritis").alias("arthritis"),
        # ---- EQ-5D-3L ----------------------------------------------------------
        level_1_3("t0_mobility").alias("eq5dMobility"),
        level_1_3("t0_self_care").alias("eq5dSelfCare"),
        level_1_3("t0_activity").alias("eq5dActivity"),
        level_1_3("t0_discomfort").alias("eq5dDiscomfort"),
        level_1_3("t0_anxiety").alias("eq5dAnxiety"),
        # ---- pre-operative context --------------------------------------------
        F.when(F.col("t0_symptom_period").between(1, 4),
               F.col("t0_symptom_period").cast("int")).alias("symptomPeriod"),
        yes_no("t0_previous_surgery").alias("previousSurgery"),
        yes_no("t0_disability").alias("disability"),
        yes_no("t0_assisted").alias("assistedCompletion"),
        F.when(F.col("living_arrangements_grouped").isin(1, 2, 4),
               F.col("living_arrangements_grouped").cast("int")).alias("livingArrangements"),
        # ---- the score ---------------------------------------------------------
        F.col("risk_poor_outcome").cast("decimal(6,5)").alias("riskPoorOutcome"),
        F.col("risk_band").alias("riskBand"),
        F.col("surgery_scheduled_date").alias("surgeryScheduledDate"),
        F.col("model_version").alias("modelVersion"),
        F.col("model_name").alias("modelName"),
    ))

# Explanations inherit the assignment so each row can filter itself.
assignments = patient_risk.select("episodeId", "assignedClinicianId", "assignedClinicianEmail")

risk_explanation = (spark.table(GOLD_EXPLAIN)
    .selectExpr(
        "episode_id     AS episodeId",
        "CAST(rank AS INT) AS rank",
        "feature",
        "feature_label  AS featureLabel",
        "CAST(feature_value AS DECIMAL(12,4)) AS featureValue",
        "CAST(contribution  AS DECIMAL(12,6)) AS contribution",
        "direction",
        "model_version  AS modelVersion",
    )
    .join(assignments, on="episodeId", how="inner"))

print(f"\npatient_risk      : {patient_risk.count():,} rows x {len(patient_risk.columns)} columns")
print(f"risk_explanation  : {risk_explanation.count():,} rows")
display(patient_risk.limit(5))


## 2b. Cohort distributions — so a number on the patient page has a denominator

`CohortStat` is the reference class. "OKS 22/48" is a fact about a patient; "OKS 22/48,
and here is where that sits among the 4,100 people on the waiting list" is a fact a
clinician can act on. The NSQIP surgical risk calculator's most-copied idea is exactly
this — showing the patient against the average patient — and it works because an
unanchored number cannot be reasoned with.

Long format, one row per (metric, bin), so a new measure is a new set of rows rather than
a schema change.

**Small-cell suppression.** A bin holding 1-4 patients is written as `0`. These are
aggregates over thousands of episodes and are not disclosive as such, but a histogram with
a bin of one is a histogram that can identify somebody, and NHS PROMs already suppresses
on the same principle upstream. The app states the rule on the chart rather than hiding it.

### Row-level security on this table, and why there is none

`CohortStat`, `ModelCurve`, `ThresholdOption` and `ModelCard` are granted `read` to any
authenticated user with **no row policy**, which is a decision rather than an oversight:

* there is no patient in them — a row is a bin label and a count, or a point on a curve;
* the entire value of the entity is the pooled denominator. Filtering the distribution to
  one clinician's list would compute it over a few dozen patients, which is statistically
  useless *and* more disclosive than the pooled version, because a clinician with eleven
  patients could read individual values off their own histogram;
* they are model-card material. A reviewing clinician who cannot see how the model
  performs is not in a position to oversee it.

The patient-scoped entities — `PatientRisk`, `RiskExplanation`, `ReviewDecision` — keep
their policies unchanged. Widening the *columns* on `PatientRisk` does not widen the
disclosure: the policy filters whole rows, so a field invisible to you today is still
invisible with thirty more columns beside it.


In [ ]:
from pyspark.sql import Row

SUPPRESSION_FLOOR = 5      # bins of 1-4 patients are written as 0

# (metric, source column, unit for the axis label, start, end, bin width)
COHORT_METRICS = [
    ("risk_poor_outcome",     F.col("riskPoorOutcome") * 100, "% probability of a poor outcome", 0,  100, 5),
    ("oks_t0_score",          F.col("oksT0Score"),            "points (0-48)",                   0,  48,  4),
    ("oks_pain_subscale",     F.col("oksPainSubscale"),       "points (0-8)",                    0,  9,   1),
    ("oks_function_subscale", F.col("oksFunctionSubscale"),   "points (0-24)",                   0,  24,  2),
    ("oks_adl_subscale",      F.col("oksAdlSubscale"),        "points (0-16)",                   0,  16,  2),
    ("comorbidity_count",     F.col("comorbidityCount"),      "conditions (0-12)",               0,  13,  1),
]

MODEL_VERSION = patient_risk.select("modelVersion").first()["modelVersion"]

stat_rows = []
for metric, expr, unit, start, end, width in COHORT_METRICS:
    values = patient_risk.select(expr.alias("v")).filter(F.col("v").isNotNull())
    n_total = values.count()
    if n_total == 0:
        print(f"  {metric}: no values, skipped")
        continue

    median = float(values.approxQuantile("v", [0.5], 0.001)[0])
    bins = int(math.ceil((end - start) / width))

    counts = dict(
        values
        .withColumn("bin", F.least(F.lit(bins - 1),
                                   F.greatest(F.lit(0), F.floor((F.col("v") - start) / width).cast("int"))))
        .groupBy("bin").count()
        .rdd.map(lambda r: (r["bin"], r["count"])).collect()
    )

    for i in range(bins):
        n = counts.get(i, 0)
        stat_rows.append(Row(
            metric=metric,
            binIndex=i,
            binStart=float(start + i * width),
            binEnd=float(start + (i + 1) * width),
            n=int(0 if n < SUPPRESSION_FLOOR else n),
            cohortSize=int(n_total),
            cohortMedian=median,
            unit=unit,
            modelVersion=str(MODEL_VERSION),
        ))
    suppressed = sum(1 for i in range(bins) if 0 < counts.get(i, 0) < SUPPRESSION_FLOOR)
    print(f"  {metric:24s} {n_total:>6,} patients, {bins:>2} bins, median {median:.1f}, "
          f"{suppressed} bin(s) suppressed")

cohort_stat = spark.createDataFrame(stat_rows)
print(f"\ncohort_stat: {cohort_stat.count():,} rows")
display(cohort_stat.filter("metric = 'oks_t0_score'").orderBy("binIndex"))


## 2c. Model artefacts — calibration, shape functions, the operating point, the card

Four questions a clinician is entitled to ask that the app previously could not answer at
all, because the evidence for them lived in notebooks nobody on a ward will ever open:

| Question | Entity | Produced by |
|---|---|---|
| Can I trust this number? | `ModelCurve` (`curveType = 'calibration'`) | notebook 43 |
| What would change it? | `ModelCurve` (`curveType = 'shape'`) | notebook 43 |
| Where is the cut-off and what does moving it cost? | `ThresholdOption` | notebook 42 |
| What is this model, measured on what? | `ModelCard` | 40 / 42 / 43 |

The app's central claim to a clinician is that *0.34 means about 34 in 100*. That is a
claim about calibration, and until now it was asserted on the patient page with nothing
behind it. Publishing the curve turns an assertion into evidence a clinician can disagree
with — which is the whole difference between decision support and an oracle.

**This cell prefers gold and computes only as a fallback.** If notebooks 42 and 43 have
already written `gold.model_calibration`, `gold.model_shape` and `gold.threshold_sweep`,
they are read straight through — that is the right architecture, because it keeps the
artefacts versioned alongside the analysis that produced them. Where they do not exist
yet, this notebook recomputes them from `@champion` and the frozen test split so that the
app is not left with four empty panels. Either way the numbers trace to one model version.


In [ ]:
import numpy as np
import pandas as pd

# Set True only when the cohort on screen is synthetic or pseudonymised. It drives a
# persistent banner in the app rather than a footnote, so it must be honest: a
# demonstration that shows invented patients without saying so is the single most
# damaging thing this delivery layer could do.
DATASET_IS_SYNTHETIC = False

DECILES = 10
FEATURE_LABELS = {          # kept in step with 50_batch_score
    "oks_t0_score":          "Overall pre-op Oxford Knee Score",
    "oks_pain_subscale":     "Pre-op pain severity",
    "oks_function_subscale": "Pre-op physical function",
    "oks_adl_subscale":      "Pre-op daily activities",
    "comorbidity_count":     "Number of long-term conditions",
    "t0_symptom_period":     "How long symptoms have lasted",
    "t0_mobility":           "Mobility (EQ-5D)",
    "t0_anxiety":            "Anxiety / depression (EQ-5D)",
    "t0_discomfort":         "Pain / discomfort (EQ-5D)",
    "t0_previous_surgery":   "Previous surgery on this knee",
    "independent_hospital":  "Treated at an independent-sector provider",
}
FEATURE_UNITS = {
    "oks_t0_score":          "points (0-48, lower is worse)",
    "oks_pain_subscale":     "points (0-8, lower is worse)",
    "oks_function_subscale": "points (0-24, lower is worse)",
    "oks_adl_subscale":      "points (0-16, lower is worse)",
    "comorbidity_count":     "conditions (0-12)",
    "t0_symptom_period":     "1 = under 1 year ... 4 = over 10 years",
}


def wilson(successes, n, z=1.96):
    """Wilson score interval -- behaves at small n, where a normal approximation does not."""
    if n == 0:
        return (0.0, 0.0)
    p = successes / n
    denom = 1 + z * z / n
    centre = p + z * z / (2 * n)
    spread = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (max(0.0, (centre - spread) / denom), min(1.0, (centre + spread) / denom))


def table_exists(name):
    try:
        return spark.catalog.tableExists(name)
    except Exception:
        return False


# ---------------------------------------------------------------------------
# Preferred path: 42/43 already wrote the artefacts to gold.
# ---------------------------------------------------------------------------
have_gold = all(table_exists(t) for t in
                ["gold.model_calibration", "gold.model_shape", "gold.threshold_sweep"])

if have_gold:
    print("Reading model artefacts from gold (written by notebooks 42 and 43).")
    model_curve = (spark.table("gold.model_calibration")
        .selectExpr("'calibration' AS curveType", "'' AS series",
                    "'Calibrated EBM' AS seriesLabel", "CAST(decile AS INT) AS pointIndex",
                    "CAST(mean_predicted AS DECIMAL(12,5)) AS x",
                    "CAST(observed AS DECIMAL(12,5)) AS y",
                    "CAST(ci_lower AS DECIMAL(12,5)) AS yLower",
                    "CAST(ci_upper AS DECIMAL(12,5)) AS yUpper",
                    "CAST(n AS INT) AS n", "'predicted probability' AS xUnit",
                    "model_version AS modelVersion")
        .unionByName(spark.table("gold.model_shape")
            .selectExpr("'shape' AS curveType", "feature AS series", "feature_label AS seriesLabel",
                        "CAST(point_index AS INT) AS pointIndex",
                        "CAST(x AS DECIMAL(12,5)) AS x", "CAST(y AS DECIMAL(12,5)) AS y",
                        "CAST(y_lower AS DECIMAL(12,5)) AS yLower",
                        "CAST(y_upper AS DECIMAL(12,5)) AS yUpper",
                        "CAST(n AS INT) AS n", "x_unit AS xUnit", "model_version AS modelVersion")))

    threshold_option = spark.table("gold.threshold_sweep").selectExpr(
        "CAST(threshold AS DECIMAL(5,4)) AS threshold",
        "CAST(flagged_per_1000 AS INT) AS flaggedPer1000",
        "CAST(correctly_flagged_per_1000 AS INT) AS correctlyFlaggedPer1000",
        "CAST(false_alarms_per_1000 AS INT) AS falseAlarmsPer1000",
        "CAST(missed_per_1000 AS INT) AS missedPer1000",
        "CAST(precision AS DECIMAL(5,4)) AS precision",
        "CAST(recall AS DECIMAL(5,4)) AS recall",
        "is_chosen AS isChosen", "rationale", "model_version AS modelVersion")

    card = spark.table("gold.model_card").toPandas().iloc[0].to_dict()

# ---------------------------------------------------------------------------
# Fallback: recompute from @champion and the frozen test split.
# ---------------------------------------------------------------------------
else:
    print("gold.model_calibration / model_shape / threshold_sweep not found — "
          "recomputing from @champion. Run 42 and 43 to make these first-class artefacts.")

    import mlflow, mlflow.sklearn
    from mlflow import MlflowClient
    from sklearn.metrics import average_precision_score, precision_score, recall_score

    REGISTERED_MODEL = "knee-poor-outcome-ebm"

    def resolve_champion(client, name):
        # Fabric's registry does not implement the alias endpoints, so the champion
        # pointer is the version tag champion='true'. Same helper as notebooks 42/43.
        try:
            return client.get_model_version_by_alias(name, "champion")
        except Exception:
            pass
        versions = list(client.search_model_versions(f"name='{name}'"))
        champs = [v for v in versions if v.tags.get("champion") == "true"]
        return max(champs or versions, key=lambda v: int(v.version))

    client = MlflowClient()
    mv = resolve_champion(client, REGISTERED_MODEL)
    model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL}/{mv.version}")

    test = spark.table("ml.test_matrix").toPandas()
    y_test = test.pop("outcome_label").astype(int).values
    X_test = test[spark.table("ml.train_matrix").drop("outcome_label").columns]
    y_prob = model.predict_proba(X_test)[:, 1]

    # ---- calibration, in deciles of predicted risk -------------------------
    order = np.argsort(y_prob)
    size = len(order) // DECILES
    cal_rows = []
    for d in range(DECILES):
        idx = order[d * size: len(order) if d == DECILES - 1 else (d + 1) * size]
        hits = int(y_test[idx].sum())
        lo, hi = wilson(hits, len(idx))
        cal_rows.append(dict(curveType="calibration", series="", seriesLabel="Calibrated EBM",
                             pointIndex=d, x=float(y_prob[idx].mean()), y=hits / len(idx),
                             yLower=lo, yUpper=hi, n=int(len(idx)),
                             xUnit="predicted probability", modelVersion=str(mv.version)))

    # ---- shape functions: the EBM's own terms ------------------------------
    def underlying_ebm(m):
        if hasattr(m, "explain_global"):
            return m
        for cc in getattr(m, "calibrated_classifiers_", []):
            est = getattr(cc, "estimator", None) or getattr(cc, "base_estimator", None)
            if hasattr(est, "explain_global"):
                return est
        return None

    shape_rows = []
    ebm = underlying_ebm(model)
    if ebm is None:
        print("  no EBM inside @champion — shape functions skipped, the app shows its empty state")
    else:
        g = ebm.explain_global()
        importances = ebm.term_importances()
        main = [i for i in np.argsort(importances)[::-1] if len(ebm.term_features_[i]) == 1][:8]
        for i in main:
            name = ebm.term_names_[i]
            d = g.data(i)
            xs, ys = d["names"], d["scores"]
            # interpret's univariate shapes come back as bin EDGES; the app plots points.
            if len(xs) == len(ys) + 1:
                xs = [(xs[k] + xs[k + 1]) / 2 for k in range(len(ys))]
            lower = d.get("lower_bounds")
            upper = d.get("upper_bounds")
            for k, (xv, yv) in enumerate(zip(xs, ys)):
                try:
                    xf = float(xv)
                except (TypeError, ValueError):
                    xf = float(k)      # categorical level: index is the position
                shape_rows.append(dict(
                    curveType="shape", series=name,
                    seriesLabel=FEATURE_LABELS.get(name, name.replace("_", " ").capitalize()),
                    pointIndex=k, x=xf, y=float(yv),
                    yLower=float(lower[k]) if lower is not None else None,
                    yUpper=float(upper[k]) if upper is not None else None,
                    n=None,
                    xUnit=FEATURE_UNITS.get(name, "feature value"),
                    modelVersion=str(mv.version)))

    model_curve = spark.createDataFrame(pd.DataFrame(cal_rows + shape_rows))

    # ---- threshold sweep, per 1,000 patients -------------------------------
    chosen_threshold = float(mv.tags.get("operating_threshold", 0.5))
    per = 1000 / len(y_test)
    sweep_rows = []
    for t in np.round(np.arange(0.05, 0.8001, 0.05), 2):
        y_hat = (y_prob >= t).astype(int)
        tp = int(((y_hat == 1) & (y_test == 1)).sum())
        fp = int(((y_hat == 1) & (y_test == 0)).sum())
        fn = int(((y_hat == 0) & (y_test == 1)).sum())
        # Round once, then add: the app's caption reads "N flags, X correct and Y
        # unnecessary", and that has to be arithmetic a clinician can check.
        correct, alarms = int(round(tp * per)), int(round(fp * per))
        sweep_rows.append(dict(
            threshold=float(t), flaggedPer1000=correct + alarms,
            correctlyFlaggedPer1000=correct, falseAlarmsPer1000=alarms,
            missedPer1000=int(round(fn * per)),
            precision=float(precision_score(y_test, y_hat, zero_division=0)),
            recall=float(recall_score(y_test, y_hat, zero_division=0)),
            isChosen=bool(abs(t - chosen_threshold) < 0.025),
            rationale=mv.tags.get("operating_threshold_rationale") if abs(t - chosen_threshold) < 0.025 else None,
            modelVersion=str(mv.version)))
    if not any(r["isChosen"] for r in sweep_rows):
        # Never leave the app without a marked operating point; say so instead.
        sweep_rows[len(sweep_rows) // 2]["isChosen"] = True
        sweep_rows[len(sweep_rows) // 2]["rationale"] = (
            f"No operating_threshold tag on model version {mv.version}; the mid-range "
            "cut-off is marked as a placeholder. Run 42_threshold_analysis to set it.")
        print("  WARNING: no operating_threshold tag — marked a placeholder cut-off.")

    threshold_option = spark.createDataFrame(pd.DataFrame(sweep_rows))

    cal = pd.DataFrame(cal_rows)
    card = dict(
        modelName=REGISTERED_MODEL,
        modelVersion=str(mv.version),
        modelType="Explainable Boosting Machine, calibrated",
        outcomeDefinition=("Poor outcome = an Oxford Knee Score gain of 7 points or fewer at six "
                           "months, i.e. at or below the minimal clinically important difference."),
        testSetSize=int(len(y_test)),
        prevalence=float(y_test.mean()),
        averagePrecision=float(average_precision_score(y_test, y_prob)),
        calibrationError=float((cal["x"] - cal["y"]).abs().mean()),
        operatingThreshold=chosen_threshold,
        dataSource="NHS PROMs knee replacement extract, via gold.knee_features in the Fabric Lakehouse.",
        datasetIsSynthetic=DATASET_IS_SYNTHETIC,
        limitations=mv.tags.get("known_limitations",
                                "See the model card in MLflow. Subgroup performance is reported in "
                                "42_threshold_analysis section 7; any group where precision falls "
                                "materially belongs here."),
        scoredAt=pd.Timestamp.utcnow().to_pydatetime().replace(tzinfo=None),
    )

model_card = spark.createDataFrame(pd.DataFrame([card]))

print(f"\nmodel_curve      : {model_curve.count():,} rows")
print(f"threshold_option : {threshold_option.count():,} rows")
print(f"model_card       : 1 row  (synthetic={card['datasetIsSynthetic']})")
display(threshold_option.orderBy("threshold"))


## 2d. Provider aggregates — one row per provider, no patient in it

`ProviderStat` is the second model-scoped aggregate. The overview page draws flag
rate and mean risk by provider so a clinician can see their site against the
service, and the Data Agent answers "how does our flag rate compare" from the same
numbers. Like `CohortStat` it carries **no row policy**: a comparison across
providers is exactly the thing a per-clinician filter would destroy, and a row is a
provider code plus counts — nothing identifies a patient.

Small-cell rules match §2b: a provider with fewer than `SUPPRESSION_FLOOR` patients
is not written at all, and any band count under the floor is written as `0`.
`flagged` uses the operating threshold recorded on the model card so the app's
"flagged" figure and this table cannot disagree.

In [ ]:
OPERATING_THRESHOLD = float(model_card.first()["operatingThreshold"])

provider_dim = spark.table("silver.provider").select(
    F.col("provider_code"), "region", "university_hospital", "independent_hospital"
)

provider_stat = (patient_risk
    .groupBy("providerCode")
    .agg(
        F.count("*").alias("patients"),
        F.avg("riskPoorOutcome").alias("meanRisk"),
        F.expr("percentile_approx(riskPoorOutcome, 0.5)").alias("medianRisk"),
        F.sum(F.when(F.col("riskPoorOutcome") >= OPERATING_THRESHOLD, 1).otherwise(0)).alias("flagged"),
        F.sum(F.when(F.col("riskBand") == "low", 1).otherwise(0)).alias("low"),
        F.sum(F.when(F.col("riskBand") == "moderate", 1).otherwise(0)).alias("moderate"),
        F.sum(F.when(F.col("riskBand") == "high", 1).otherwise(0)).alias("high"),
        F.sum(F.when(F.col("riskBand") == "very_high", 1).otherwise(0)).alias("veryHigh"),
        F.expr("percentile_approx(oksT0Score, 0.5)").alias("medianOks"),
        F.first("providerType").alias("providerType"),
        F.first("region").alias("region"),
    )
    .filter(F.col("patients") >= SUPPRESSION_FLOOR)
)

# Band counts under the floor -> 0, matching CohortStat.
for c in ["low", "moderate", "high", "veryHigh"]:
    provider_stat = provider_stat.withColumn(
        c, F.when(F.col(c) < SUPPRESSION_FLOOR, F.lit(0)).otherwise(F.col(c)).cast("int"))

provider_stat = (provider_stat
    .withColumn("patients", F.col("patients").cast("int"))
    .withColumn("flagged", F.col("flagged").cast("int"))
    .withColumn("meanRisk", F.col("meanRisk").cast("decimal(6,5)"))
    .withColumn("medianRisk", F.col("medianRisk").cast("decimal(6,5)"))
    .withColumn("medianOks", F.col("medianOks").cast("decimal(6,2)"))
    .withColumn("modelVersion", F.lit(str(MODEL_VERSION)))
    .select("providerCode", "providerType", "region", "patients", "meanRisk", "medianRisk",
            "flagged", "low", "moderate", "high", "veryHigh", "medianOks", "modelVersion")
)

print(f"provider_stat: {provider_stat.count():,} providers at threshold {OPERATING_THRESHOLD:.2f}")
display(provider_stat.orderBy(F.col("flagged") / F.col("patients"), ascending=False).limit(15))

## 3. Connect, and check what `rayfin up` actually created

Before writing anything, list the tables. Rayfin derives table names from the entity
class names with `pluralize.plural()`, preserving PascalCase
(`rayfin-lib/dist/utils/EntityNameResolver.js`), so `PatientRisk` becomes `PatientRisks`
-- not `patient_risks`. Column names come straight from the TypeScript property names,
so they are camelCase in SQL too.

Rather than trust that derivation, read it off the database. If the names below differ
from `TABLES`, change `TABLES` and not the entity classes -- the app is already bound to
whatever DAB generated.

In [ ]:
from notebookutils import mssparkutils

# The notebook's own Entra token. Nothing is written down, and the token carries
# the identity of whoever is running the notebook -- which is also what makes the
# RLS check in section 5 meaningful.
#
# ActiveDirectoryMSI only works when the notebook runs under a managed identity.
# Driving it by hand, the access token is the reliable path.
SQL_TOKEN = mssparkutils.credentials.getToken("https://database.windows.net/")

# --- TLS to a Fabric SQL database -----------------------------------------
#
# The connection lands on a gateway and is then *redirected* to a worker node,
# and the worker presents its own certificate:
#
#   connecting to  <db>.database.fabric.microsoft.com
#   cert says      tr40862.westeurope1-a.worker.database.windows.net
#
# So strict validation fails with "could not establish a secure connection ...
# Failed to validate the server name". Pinning `hostNameInCertificate` does not
# fix it: the worker id is not stable, so there is no name to pin.
#
# This tries strict first and falls back only on that specific failure, printing
# what it did. If the driver or the service starts presenting a matching chain,
# the notebook silently goes back to validating — nothing here has to be undone.
#
# What the fallback gives up: server *identity* validation. The channel is still
# encrypted and still authenticated by the Entra token below, and the hop stays
# inside the Fabric estate. Worth saying out loud rather than leaving as a quiet
# `trustServerCertificate=true` in a talk about governance.

def jdbc_url(trust_server_certificate: bool) -> str:
    return (
        f"jdbc:sqlserver://{SQL_SERVER}:1433;"
        f"database={SQL_DATABASE};"
        "encrypt=true;"
        f"trustServerCertificate={'true' if trust_server_certificate else 'false'};"
        "loginTimeout=60;"
    )


def _read(url, query):
    return (spark.read.format("jdbc")
            .option("url", url)
            .option("accessToken", SQL_TOKEN)
            .option("query", query)
            .load())


PROBE = "SELECT 1 AS ok"
try:
    JDBC_URL = jdbc_url(trust_server_certificate=False)
    _read(JDBC_URL, PROBE).collect()
    print("TLS: server certificate validated.")
except Exception as exc:
    if "validate the server name" not in str(exc) and "Secure Sockets Layer" not in str(exc):
        raise
    JDBC_URL = jdbc_url(trust_server_certificate=True)
    _read(JDBC_URL, PROBE).collect()
    print("TLS: WARNING -- server certificate NOT validated (gateway redirects to a\n"
          "     worker node whose certificate name does not match). The connection is\n"
          "     still encrypted and still authenticated by the Entra token.")


def read_sql(query):
    return _read(JDBC_URL, query)


catalog = read_sql("""
    SELECT TABLE_SCHEMA, TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE'
""")
display(catalog)

existing = {f"{r.TABLE_SCHEMA}.{r.TABLE_NAME}" for r in catalog.collect()}
print(f"{len(existing)} tables in {SQL_DATABASE}")

TABLES = {
    "patient_risk":      "dbo.PatientRisks",
    "risk_explanation":  "dbo.RiskExplanations",
    "review_decision":   "dbo.ReviewDecisions",
    # Added with the redesigned app. Rayfin pluralises the entity class name and
    # preserves PascalCase, so CohortStat -> CohortStats. If `rayfin up db apply`
    # has not run since the entities were added, these will be missing and the
    # check below names them rather than failing several thousand rows into a write.
    "cohort_stat":       "dbo.CohortStats",
    "provider_stat":     "dbo.ProviderStats",
    "model_curve":       "dbo.ModelCurves",
    "threshold_option":  "dbo.ThresholdOptions",
    "model_card":        "dbo.ModelCards",
}

missing = [t for t in TABLES.values() if t not in existing]
if missing:
    raise RuntimeError(
        f"Not in the database: {missing}. Either `rayfin up db apply` has not run, "
        f"or the names differ -- pick the right ones from the list above and edit "
        f"TABLES. Found: {sorted(existing)}")

for logical, table in TABLES.items():
    # No ORDER BY here: the JDBC `query` option wraps the statement as a derived
    # table, and T-SQL rejects ORDER BY inside one unless TOP/OFFSET is also
    # present. Select the ordinal and sort in Python instead -- same result,
    # and it does not depend on a driver quirk.
    cols = read_sql(f"""
        SELECT ORDINAL_POSITION, COLUMN_NAME, DATA_TYPE, IS_NULLABLE
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_NAME = '{table.split('.')[1]}'
    """)
    ordered = sorted(cols.collect(), key=lambda r: r.ORDINAL_POSITION)
    print(f"\n{table}:")
    print("  " + ", ".join(r.COLUMN_NAME for r in ordered))


## 4. Write


In [ ]:
def destination_columns(table):
    """The columns the destination table actually has, lower-cased for matching."""
    rows = read_sql(f"""
        SELECT COLUMN_NAME
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_NAME = '{table.split('.')[1]}'
    """).collect()
    return {r.COLUMN_NAME.lower(): r.COLUMN_NAME for r in rows}


def align_to_destination(df, table):
    """Drop columns the destination does not have, and say so.

    The app's entity classes and this database drift whenever `rayfin up db apply`
    has not been run since a column was added -- which is the normal state for a
    few minutes after a schema change, and an easy thing to hit mid-rehearsal.
    Spark's JDBC writer fails that with a wall of StructType text that does not
    name the fix.

    Dropping is safe in one direction only. An *extra* column here is a feature
    the app cannot use yet, so the sync proceeds without it and prints what to
    run. A *missing* required column would be data loss, and the writer still
    raises for that.
    """
    have = destination_columns(table)
    extra = [c for c in df.columns if c.lower() not in have]
    if extra:
        print(f"  {table}: destination has no {', '.join(extra)} -- writing without "
              f"{'it' if len(extra) == 1 else 'them'}.")
        print(f"    Run `npx rayfin up db apply` to add {'it' if len(extra) == 1 else 'them'}, "
              f"then re-run this notebook.")
        df = df.drop(*extra)
    return df


def write_jdbc(df, table, mode="overwrite"):
    df = align_to_destination(df, table)
    # `truncate` keeps the table and its grants in place rather than dropping and
    # recreating it -- a recreated table loses whatever DAB configured on it.
    (df.write
       .format("jdbc")
       .option("url", JDBC_URL)
       .option("accessToken", SQL_TOKEN)
       .option("dbtable", table)
       .option("batchsize", 5000)
       .option("truncate", "true")
       .mode(mode)
       .save())
    print(f"wrote {table}: {df.count():,} rows")


# The column printout above is the authority on names and order. A mismatch here
# surfaces as a type error several thousand rows into the write, not at the start.
write_jdbc(patient_risk,     TABLES["patient_risk"])
write_jdbc(risk_explanation, TABLES["risk_explanation"])

# The model-scoped tables. These are aggregates and model documentation, carry no
# patient rows, and are declared without a row policy -- see section 2b for why
# filtering them would be both useless and more disclosive than not.
write_jdbc(cohort_stat,      TABLES["cohort_stat"])
write_jdbc(provider_stat,    TABLES["provider_stat"])
write_jdbc(model_curve,      TABLES["model_curve"])
write_jdbc(threshold_option, TABLES["threshold_option"])
write_jdbc(model_card,       TABLES["model_card"])

print("\nReviewDecisions is deliberately left alone: the app writes it, this "
      "notebook does not. Truncating it would delete recorded clinical decisions.")


## 5. Verify the row-level security actually bites

The most valuable ten seconds of rehearsal you will spend. This is the *data-side* check;
the real proof is signing into the app as each clinician and seeing two different lists.
Do both.

In [ ]:
per_clinician = (patient_risk
    .groupBy("assignedClinicianId", "assignedClinicianEmail")
    .agg(F.count("*").alias("patients"),
         F.countDistinct("providerCode").alias("providers"),
         F.round(F.avg("riskPoorOutcome") * 100, 1).alias("mean_risk_pct")))
display(per_clinician)

total = patient_risk.count()
covered = per_clinician.agg(F.sum("patients")).first()[0]
assert covered == total, f"{total - covered} patients have no clinician assignment"
print(f"All {total:,} patients are assigned. No row is unreachable — and none is universally reachable.")

# The security model is two-tier, and the split should be stated out loud rather than
# inferred from four entity files. Patient-scoped entities filter rows against a claim;
# model-scoped entities do not, because they contain no patient.
print("""
Row-level security, by entity:

  PatientRisk       filtered   claims.sub == assignedClinicianId
                               OR claims.email == assignedClinicianEmail
                               (or role == governance)
  RiskExplanation   filtered   same, denormalised onto the row
  ReviewDecision    filtered   claims.sub == clinicianId          (or role == governance)

  CohortStat        open       aggregate bins, small cells suppressed, no patient in it
  ProviderStat      open       per-provider aggregates, providers under the floor omitted
  ModelCurve        open       calibration and shape functions -- the model, not a patient
  ThresholdOption   open       test-set counts per 1,000
  ModelCard         open       model provenance and headline metrics

"Open" means read for any authenticated caller. Data API Builder still requires a valid
session; nothing here is anonymous. Enforcement for all eight is in DAB, never in the
frontend -- open devtools on stage, rewrite the GraphQL query, and PatientRisk still
returns only your own patients.
""")

---

Run order for a full rebuild: **10 → 20 → 30 → 40 → 50 → 60**, then `npx rayfin up` in
`fabric/rayfin-clinician-app/`.